# DailyETLMain — Databricks Migration

Generated from SSIS package by the migration agent.

| Field | Value |
| --- | --- |
| Steps | 13 |
| Unresolved | 0 |
| Planner | deterministic |


In [ ]:
# Databricks Job parameters
# When run via a Databricks Job, each task passes
# step_id to execute only its portion.
# When run interactively, step_id="all" runs
# every step in dependency order.
dbutils.widgets.text('step_id', 'all', 'Step ID to execute')
dbutils.widgets.text('catalog', 'main', 'Target catalog')
_step_id = dbutils.widgets.get('step_id')
_catalog = dbutils.widgets.get('catalog')


In [ ]:
# ------------------------------------------------------------
# Setup: ETL control-table helpers
# ------------------------------------------------------------
from datetime import datetime, timezone

_ETL_CUTOFF_TABLE = f"{_catalog}.Integration.ETL_Cutoff"
_LINEAGE_TABLE = f"{_catalog}.Integration.Lineage"


def _control_table_key(table_name: str) -> str:
    """Control tables are keyed by schema.table (SQL Server
    semantics), without the catalog prefix."""
    if table_name.startswith(_catalog + "."):
        return table_name.split(".", 1)[1]
    return table_name


def _fmt_cutoff(value) -> str:
    """Format a datetime for safe SQL string embedding."""
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d %H:%M:%S.%f")
    return str(value)


# SSIS: Calculate ETL Cutoff Time + Trim Any Milliseconds
new_cutoff = datetime.now(timezone.utc).replace(
    tzinfo=None, microsecond=0
)


def get_last_etl_cutoff(table_name: str) -> datetime:
    """Integration.GetLastETLCutoffTime: MAX(Cutoff Time) for
    the table; 0001-01-01 when the table has no row yet."""
    row = spark.sql(
        "SELECT MAX(`Cutoff Time`) AS cutoff "
        f"FROM {_ETL_CUTOFF_TABLE} "
        f"WHERE `Table Name` = '{_control_table_key(table_name)}'"
    ).collect()[0]
    if row["cutoff"] is None:
        return datetime(1, 1, 1)
    return row["cutoff"]


def get_lineage_key(table_name: str, cutoff_time) -> int:
    """Integration.GetLineageKey: start a lineage record and
    return its key."""
    spark.sql(
        f"INSERT INTO {_LINEAGE_TABLE} "
        "(`Data Load Started`, `Table Name`, `Was Successful`, "
        "`Source System Cutoff Time`) "
        "VALUES (current_timestamp(), "
        f"'{_control_table_key(table_name)}', false, "
        f"'{_fmt_cutoff(cutoff_time)}')"
    )
    row = spark.sql(
        "SELECT `Lineage Key` AS lineage_key "
        f"FROM {_LINEAGE_TABLE} "
        f"WHERE `Table Name` = '{_control_table_key(table_name)}' "
        "ORDER BY `Data Load Started` DESC, `Lineage Key` DESC "
        "LIMIT 1"
    ).collect()[0]
    return row["lineage_key"]


def complete_lineage(lineage_key: int) -> None:
    """Mark the lineage record as successfully completed."""
    spark.sql(
        f"UPDATE {_LINEAGE_TABLE} "
        "SET `Was Successful` = true, "
        "`Data Load Completed` = current_timestamp() "
        f"WHERE `Lineage Key` = {lineage_key}"
    )


def update_etl_cutoff(table_name: str, cutoff_time) -> None:
    """Integration.RecordLastETLCutoffTime: upsert the cutoff."""
    spark.sql(
        f"MERGE INTO {_ETL_CUTOFF_TABLE} AS tgt "
        "USING (SELECT "
        f"'{_control_table_key(table_name)}' AS table_name, "
        f"'{_fmt_cutoff(cutoff_time)}' AS cutoff) AS src "
        "ON tgt.`Table Name` = src.table_name "
        "WHEN MATCHED THEN UPDATE SET "
        "tgt.`Cutoff Time` = src.cutoff "
        "WHEN NOT MATCHED THEN INSERT "
        "(`Table Name`, `Cutoff Time`) "
        "VALUES (src.table_name, src.cutoff)"
    )


## Load City Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_city_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetCityUpdates` |
| Target Objects | `Integration.City_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load City Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetCityUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedCityData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_city_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.City_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.City_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.City_Staging (`WWI City ID`, City, `State Province`, Country, Continent, `Sales Territory`, Region, Subregion, `Location`, `Latest Recorded Population`, `Valid From`, `Valid To`)
        SELECT c.CityID, c.CityName, sp.StateProvinceName, sp.CountryName, sp.Continent, sp.SalesTerritory, sp.Region, sp.Subregion, c.Location, c.LatestRecordedPopulation, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Application.Cities c
        INNER JOIN {_catalog}.Application.StateProvinces sp ON c.StateProvinceID = sp.StateProvinceID
        WHERE c.LastEditedWhen > '{last_cutoff}' AND c.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.City AS tgt
        USING (SELECT `WWI City ID`, City, `State Province`, Country, Continent, `Sales Territory`, Region, Subregion, `Location`, `Latest Recorded Population`, `Valid From`, `Valid To` FROM {_catalog}.Integration.City_Staging) AS src
        ON (tgt.`WWI City ID` = src.`WWI City ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.City IS DISTINCT FROM tgt.City OR src.`State Province` IS DISTINCT FROM tgt.`State Province` OR src.Country IS DISTINCT FROM tgt.Country OR src.Continent IS DISTINCT FROM tgt.Continent OR src.`Sales Territory` IS DISTINCT FROM tgt.`Sales Territory` OR src.Region IS DISTINCT FROM tgt.Region OR src.Subregion IS DISTINCT FROM tgt.Subregion OR src.`Location` IS DISTINCT FROM tgt.`Location` OR src.`Latest Recorded Population` IS DISTINCT FROM tgt.`Latest Recorded Population`)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI City ID`, City, `State Province`, Country, Continent, `Sales Territory`, Region, Subregion, `Location`, `Latest Recorded Population`, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI City ID`, src.City, src.`State Province`, src.Country, src.Continent, src.`Sales Territory`, src.Region, src.Subregion, src.`Location`, src.`Latest Recorded Population`, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Customer Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_customer_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetCustomerUpdates` |
| Target Objects | `Integration.Customer_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Customer Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetCustomerUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedCustomerData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_customer_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Customer_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Customer_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Customer_Staging (`WWI Customer ID`, Customer, `Bill To Customer`, Category, `Buying Group`, `Primary Contact`, `Postal Code`, `Valid From`, `Valid To`)
        SELECT c.CustomerID, c.CustomerName, bt.CustomerName, c.CustomerCategoryName, bg.BuyingGroupName, p.FullName, c.DeliveryPostalCode, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Sales.Customers c
        LEFT OUTER JOIN {_catalog}.Sales.Customers bt ON c.BillToCustomerID = bt.CustomerID
        LEFT OUTER JOIN {_catalog}.Sales.BuyingGroups bg ON c.BuyingGroupID = bg.BuyingGroupID
        LEFT OUTER JOIN {_catalog}.Application.People p ON c.PrimaryContactPersonID = p.PersonID
        WHERE c.LastEditedWhen > '{last_cutoff}' AND c.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Customer"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Customer"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.Customer AS tgt
        USING (SELECT `WWI Customer ID`, Customer, `Bill To Customer`, Category, `Buying Group`, `Primary Contact`, `Postal Code`, `Valid From`, `Valid To` FROM {_catalog}.Integration.Customer_Staging) AS src
        ON (tgt.`WWI Customer ID` = src.`WWI Customer ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.Customer IS DISTINCT FROM tgt.Customer OR src.`Bill To Customer` IS DISTINCT FROM tgt.`Bill To Customer` OR src.Category IS DISTINCT FROM tgt.Category OR src.`Buying Group` IS DISTINCT FROM tgt.`Buying Group` OR src.`Primary Contact` IS DISTINCT FROM tgt.`Primary Contact` OR src.`Postal Code` IS DISTINCT FROM tgt.`Postal Code`)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Customer ID`, Customer, `Bill To Customer`, Category, `Buying Group`, `Primary Contact`, `Postal Code`, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Customer ID`, src.Customer, src.`Bill To Customer`, src.Category, src.`Buying Group`, src.`Primary Contact`, src.`Postal Code`, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Employee Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_employee_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetEmployeeUpdates` |
| Target Objects | `Integration.Employee_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Employee Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetEmployeeUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedEmployeeData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_employee_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Employee_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Employee_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Employee_Staging (`WWI Employee ID`, Employee, `Preferred Name`, `Is Salesperson`, Photo, `Valid From`, `Valid To`)
        SELECT p.PersonID, p.FullName, p.PreferredName, p.IsSalesperson, p.Photo, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Application.People p
        WHERE p.IsEmployee = 1 AND p.LastEditedWhen > '{last_cutoff}' AND p.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Employee"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Employee"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.Employee AS tgt
        USING (SELECT `WWI Employee ID`, Employee, `Preferred Name`, `Is Salesperson`, Photo, `Valid From`, `Valid To` FROM {_catalog}.Integration.Employee_Staging) AS src
        ON (tgt.`WWI Employee ID` = src.`WWI Employee ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.Employee IS DISTINCT FROM tgt.Employee OR src.`Preferred Name` IS DISTINCT FROM tgt.`Preferred Name` OR src.`Is Salesperson` IS DISTINCT FROM tgt.`Is Salesperson` OR src.Photo IS DISTINCT FROM tgt.Photo)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Employee ID`, Employee, `Preferred Name`, `Is Salesperson`, Photo, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Employee ID`, src.Employee, src.`Preferred Name`, src.`Is Salesperson`, src.Photo, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Movement Fact

| Field | Value |
| --- | --- |
| Step ID | `load_movement_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetMovementUpdates` |
| Target Objects | `Integration.Movement_Staging` |


In [ ]:
if _step_id in ("all", "load_movement_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Movement_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Movement_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Movement_Staging
    SELECT CAST(sit.TransactionOccurredWhen AS date) AS `Date Key`,
               sit.StockItemTransactionID AS `WWI Stock Item Transaction ID`,
               sit.InvoiceID AS `WWI Invoice ID`,
               sit.PurchaseOrderID AS `WWI Purchase Order ID`,
               CAST(sit.Quantity AS int) AS Quantity,
               sit.StockItemID AS `WWI Stock Item ID`,
               sit.CustomerID AS `WWI Customer ID`,
               sit.SupplierID AS `WWI Supplier ID`,
               sit.TransactionTypeID AS `WWI Transaction Type ID`,
               sit.TransactionOccurredWhen AS `Transaction Occurred When`
        FROM {_catalog}.Warehouse.StockItemTransactions AS sit
        WHERE sit.LastEditedWhen > '{last_cutoff}'
        AND sit.LastEditedWhen <= '{new_cutoff}'
        ORDER BY sit.StockItemTransactionID
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"


    # [merge_target] (MERGE)
    merge_target_sql = f"""
    MERGE INTO {_catalog}.Fact.Movement AS tgt
        USING (SELECT `Date Key`, `WWI Stock Item Transaction ID`, `WWI Invoice ID`, `WWI Purchase Order ID`, `Quantity`, `WWI Stock Item ID`, `WWI Customer ID`, `WWI Supplier ID`, `WWI Transaction Type ID`, `Last Modified When` FROM {_catalog}.Integration.Movement_Staging) AS src
        ON (tgt.`WWI Stock Item Transaction ID` = src.`WWI Stock Item Transaction ID`)
        WHEN MATCHED AND (src.`Date Key` IS DISTINCT FROM tgt.`Date Key` OR src.`Quantity` IS DISTINCT FROM tgt.`Quantity` OR src.`WWI Stock Item ID` IS DISTINCT FROM tgt.`WWI Stock Item ID` OR src.`WWI Customer ID` IS DISTINCT FROM tgt.`WWI Customer ID` OR src.`WWI Supplier ID` IS DISTINCT FROM tgt.`WWI Supplier ID` OR src.`WWI Transaction Type ID` IS DISTINCT FROM tgt.`WWI Transaction Type ID`)
        THEN UPDATE SET tgt.`Date Key` = src.`Date Key`, tgt.`Quantity` = src.`Quantity`, tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`, tgt.`WWI Customer ID` = src.`WWI Customer ID`, tgt.`WWI Supplier ID` = src.`WWI Supplier ID`, tgt.`WWI Transaction Type ID` = src.`WWI Transaction Type ID`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`Date Key`, `WWI Stock Item Transaction ID`, `WWI Invoice ID`, `WWI Purchase Order ID`, `Quantity`, `WWI Stock Item ID`, `WWI Customer ID`, `WWI Supplier ID`, `WWI Transaction Type ID`, `Lineage Key`)
        VALUES (src.`Date Key`, src.`WWI Stock Item Transaction ID`, src.`WWI Invoice ID`, src.`WWI Purchase Order ID`, src.`Quantity`, src.`WWI Stock Item ID`, src.`WWI Customer ID`, src.`WWI Supplier ID`, src.`WWI Transaction Type ID`, {lineage_key})
    """
    spark.sql(merge_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Order Fact

| Field | Value |
| --- | --- |
| Step ID | `load_order_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetOrderUpdates` |
| Target Objects | `Integration.Order_Staging` |


In [ ]:
if _step_id in ("all", "load_order_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Order_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Order_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Order_Staging
    SELECT CAST(o.OrderDate AS date) AS `Order Date Key`,
               CAST(ol.PickingCompletedWhen AS date) AS `Picked Date Key`,
               o.OrderID AS `WWI Order ID`,
               o.BackorderOrderID AS `WWI Backorder ID`,
               ol.`Description`,
               pt.PackageTypeName AS Package,
               ol.Quantity AS Quantity,
               ol.UnitPrice AS `Unit Price`,
               ol.TaxRate AS `Tax Rate`,
               ROUND(ol.Quantity * ol.UnitPrice, 2) AS `Total Excluding Tax`,
               ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Tax Amount`,
               ROUND(ol.Quantity * ol.UnitPrice, 2) + ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Total Including Tax`,
               c.DeliveryCityID AS `WWI City ID`,
               c.CustomerID AS `WWI Customer ID`,
               ol.StockItemID AS `WWI Stock Item ID`,
               o.SalespersonPersonID AS `WWI Salesperson ID`,
               o.PickedByPersonID AS `WWI Picker ID`,
               CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END AS `Last Modified When`
        FROM {_catalog}.Sales.Orders AS o
        INNER JOIN {_catalog}.Sales.OrderLines AS ol
        ON o.OrderID = ol.OrderID
        INNER JOIN {_catalog}.Warehouse.PackageTypes AS pt
        ON ol.PackageTypeID = pt.PackageTypeID
        INNER JOIN {_catalog}.Sales.Customers AS c
        ON c.CustomerID = o.CustomerID
        WHERE CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END > '{last_cutoff}'
        AND CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END <= '{new_cutoff}'
        ORDER BY o.OrderID
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


    # [delete_existing_target] (DELETE)
    delete_target_sql = f"""
    DELETE FROM {_catalog}.Fact.Order
    WHERE `WWI Order ID` IN (SELECT `WWI Order ID` FROM {_catalog}.Integration.Order_Staging)
    """
    spark.sql(delete_target_sql)


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Fact.`Order` AS tgt
        USING (SELECT `Order Date Key`, `Picked Date Key`, `WWI Order ID`, `WWI Backorder ID`, `Description`, `Package`, `Quantity`, `Unit Price`, `Tax Rate`, `Total Excluding Tax`, `Tax Amount`, `Total Including Tax`, `WWI City ID`, `WWI Customer ID`, `WWI Stock Item ID`, `WWI Salesperson ID`, `WWI Picker ID`, `Last Modified When` FROM {_catalog}.Integration.Order_Staging) AS src
        ON (tgt.`WWI Order ID` = src.`WWI Order ID`)
        WHEN MATCHED AND (src.`Order Date Key` IS DISTINCT FROM tgt.`Order Date Key` OR src.`Picked Date Key` IS DISTINCT FROM tgt.`Picked Date Key` OR src.`Description` IS DISTINCT FROM tgt.`Description` OR src.`Package` IS DISTINCT FROM tgt.`Package` OR src.`Quantity` IS DISTINCT FROM tgt.`Quantity` OR src.`Unit Price` IS DISTINCT FROM tgt.`Unit Price` OR src.`Tax Rate` IS DISTINCT FROM tgt.`Tax Rate` OR src.`Total Excluding Tax` IS DISTINCT FROM tgt.`Total Excluding Tax` OR src.`Tax Amount` IS DISTINCT FROM tgt.`Tax Amount` OR src.`Total Including Tax` IS DISTINCT FROM tgt.`Total Including Tax` OR src.`WWI City ID` IS DISTINCT FROM tgt.`WWI City ID` OR src.`WWI Customer ID` IS DISTINCT FROM tgt.`WWI Customer ID` OR src.`WWI Stock Item ID` IS DISTINCT FROM tgt.`WWI Stock Item ID` OR src.`WWI Salesperson ID` IS DISTINCT FROM tgt.`WWI Salesperson ID` OR src.`WWI Picker ID` IS DISTINCT FROM tgt.`WWI Picker ID`)
        THEN UPDATE SET tgt.`Order Date Key` = src.`Order Date Key`, tgt.`Picked Date Key` = src.`Picked Date Key`, tgt.`Description` = src.`Description`, tgt.`Package` = src.`Package`, tgt.`Quantity` = src.`Quantity`, tgt.`Unit Price` = src.`Unit Price`, tgt.`Tax Rate` = src.`Tax Rate`, tgt.`Total Excluding Tax` = src.`Total Excluding Tax`, tgt.`Tax Amount` = src.`Tax Amount`, tgt.`Total Including Tax` = src.`Total Including Tax`, tgt.`WWI City ID` = src.`WWI City ID`, tgt.`WWI Customer ID` = src.`WWI Customer ID`, tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`, tgt.`WWI Salesperson ID` = src.`WWI Salesperson ID`, tgt.`WWI Picker ID` = src.`WWI Picker ID`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`Order Date Key`, `Picked Date Key`, `WWI Order ID`, `WWI Backorder ID`, `Description`, `Package`, `Quantity`, `Unit Price`, `Tax Rate`, `Total Excluding Tax`, `Tax Amount`, `Total Including Tax`, `WWI City ID`, `WWI Customer ID`, `WWI Stock Item ID`, `WWI Salesperson ID`, `WWI Picker ID`, `Lineage Key`)
        VALUES (src.`Order Date Key`, src.`Picked Date Key`, src.`WWI Order ID`, src.`WWI Backorder ID`, src.`Description`, src.`Package`, src.`Quantity`, src.`Unit Price`, src.`Tax Rate`, src.`Total Excluding Tax`, src.`Tax Amount`, src.`Total Including Tax`, src.`WWI City ID`, src.`WWI Customer ID`, src.`WWI Stock Item ID`, src.`WWI Salesperson ID`, src.`WWI Picker ID`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Payment Method Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_payment_method_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetPaymentMethodUpdates` |
| Target Objects | `Integration.PaymentMethod_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Payment Method Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetPaymentMethodUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedPaymentMethodData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_payment_method_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.PaymentMethod_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.PaymentMethod_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.PaymentMethod_Staging (`WWI Payment Method ID`, `Payment Method`, `Valid From`, `Valid To`)
        SELECT pm.PaymentMethodID, pm.PaymentMethodName, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Application.PaymentMethods pm
        WHERE pm.LastEditedWhen > '{last_cutoff}' AND pm.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Payment Method"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Payment Method"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.`Payment Method` AS tgt
        USING (SELECT `WWI Payment Method ID`, `Payment Method`, `Valid From`, `Valid To` FROM {_catalog}.Integration.PaymentMethod_Staging) AS src
        ON (tgt.`WWI Payment Method ID` = src.`WWI Payment Method ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.`Payment Method` IS DISTINCT FROM tgt.`Payment Method`)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Payment Method ID`, `Payment Method`, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Payment Method ID`, src.`Payment Method`, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Purchase Fact

| Field | Value |
| --- | --- |
| Step ID | `load_purchase_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetPurchaseUpdates` |
| Target Objects | `Integration.Purchase_Staging` |


In [ ]:
if _step_id in ("all", "load_purchase_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Purchase_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Purchase_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Purchase_Staging
    SELECT CAST(po.OrderDate AS date) AS `Date Key`,
               po.PurchaseOrderID AS `WWI Purchase Order ID`,
               pol.OrderedOuters AS `Ordered Outers`,
               pol.OrderedOuters * si.QuantityPerOuter AS `Ordered Quantity`,
               pol.ReceivedOuters AS `Received Outers`,
               pt.PackageTypeName AS Package,
               pol.IsOrderLineFinalized AS `Is Order Finalized`,
               po.SupplierID AS `WWI Supplier ID`,
               pol.StockItemID AS `WWI Stock Item ID`,
               CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END AS `Last Modified When`
        FROM {_catalog}.Purchasing.PurchaseOrders AS po
        INNER JOIN {_catalog}.Purchasing.PurchaseOrderLines AS pol
        ON po.PurchaseOrderID = pol.PurchaseOrderID
        INNER JOIN {_catalog}.Warehouse.StockItems AS si
        ON pol.StockItemID = si.StockItemID
        INNER JOIN {_catalog}.Warehouse.PackageTypes AS pt
        ON pol.PackageTypeID = pt.PackageTypeID
        WHERE CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END > '{last_cutoff}'
        AND CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END <= '{new_cutoff}'
        ORDER BY po.PurchaseOrderID
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Supplier", "Dimension.Stock Item"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Supplier", "Dimension.Stock Item"


    # [delete_existing_target] (DELETE)
    delete_target_sql = f"""
    DELETE FROM {_catalog}.Fact.Purchase
    WHERE `WWI Purchase Order ID` IN (SELECT `WWI Purchase Order ID` FROM {_catalog}.Integration.Purchase_Staging)
    """
    spark.sql(delete_target_sql)


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Fact.Purchase AS tgt
        USING (SELECT `Date Key`, `WWI Purchase Order ID`, `Ordered Outers`, `Ordered Quantity`, `Received Outers`, `Package`, `Is Order Line Finalized`, `WWI Supplier ID`, `WWI Stock Item ID`, `Last Modified When` FROM {_catalog}.Integration.Purchase_Staging) AS src
        ON (tgt.`WWI Purchase Order ID` = src.`WWI Purchase Order ID` AND tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`)
        WHEN MATCHED AND (src.`Date Key` IS DISTINCT FROM tgt.`Date Key` OR src.`Ordered Outers` IS DISTINCT FROM tgt.`Ordered Outers` OR src.`Ordered Quantity` IS DISTINCT FROM tgt.`Ordered Quantity` OR src.`Received Outers` IS DISTINCT FROM tgt.`Received Outers` OR src.`Package` IS DISTINCT FROM tgt.`Package` OR src.`Is Order Line Finalized` IS DISTINCT FROM tgt.`Is Order Line Finalized` OR src.`WWI Supplier ID` IS DISTINCT FROM tgt.`WWI Supplier ID` OR src.`WWI Stock Item ID` IS DISTINCT FROM tgt.`WWI Stock Item ID`)
        THEN UPDATE SET tgt.`Date Key` = src.`Date Key`, tgt.`Ordered Outers` = src.`Ordered Outers`, tgt.`Ordered Quantity` = src.`Ordered Quantity`, tgt.`Received Outers` = src.`Received Outers`, tgt.`Package` = src.`Package`, tgt.`Is Order Line Finalized` = src.`Is Order Line Finalized`, tgt.`WWI Supplier ID` = src.`WWI Supplier ID`, tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`Date Key`, `WWI Purchase Order ID`, `Ordered Outers`, `Ordered Quantity`, `Received Outers`, `Package`, `Is Order Line Finalized`, `WWI Supplier ID`, `WWI Stock Item ID`, `Lineage Key`)
        VALUES (src.`Date Key`, src.`WWI Purchase Order ID`, src.`Ordered Outers`, src.`Ordered Quantity`, src.`Received Outers`, src.`Package`, src.`Is Order Line Finalized`, src.`WWI Supplier ID`, src.`WWI Stock Item ID`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Sale Fact

| Field | Value |
| --- | --- |
| Step ID | `load_sale_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetSaleUpdates` |
| Target Objects | `Integration.Sale_Staging` |


In [ ]:
if _step_id in ("all", "load_sale_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Sale_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Sale_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Sale_Staging
    SELECT CAST(i.InvoiceDate AS date) AS `Invoice Date Key`,
               CAST(i.ConfirmedDeliveryTime AS date) AS `Delivery Date Key`,
               i.InvoiceID AS `WWI Invoice ID`,
               il.`Description`,
               pt.PackageTypeName AS Package,
               il.Quantity,
               il.UnitPrice AS `Unit Price`,
               il.TaxRate AS `Tax Rate`,
               il.ExtendedPrice - il.TaxAmount AS `Total Excluding Tax`,
               il.TaxAmount AS `Tax Amount`,
               il.LineProfit AS Profit,
               il.ExtendedPrice AS `Total Including Tax`,
               CASE WHEN si.IsChillerStock = 0 THEN il.Quantity ELSE 0 END AS `Total Dry Items`,
               CASE WHEN si.IsChillerStock <> 0 THEN il.Quantity ELSE 0 END AS `Total Chiller Items`,
               c.DeliveryCityID AS `WWI City ID`,
               i.CustomerID AS `WWI Customer ID`,
               i.BillToCustomerID AS `WWI Bill To Customer ID`,
               il.StockItemID AS `WWI Stock Item ID`,
               i.SalespersonPersonID AS `WWI Saleperson ID`,
               CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END AS `Last Modified When`
        FROM {_catalog}.Sales.Invoices AS i
        INNER JOIN {_catalog}.Sales.InvoiceLines AS il
        ON i.InvoiceID = il.InvoiceID
        INNER JOIN {_catalog}.Warehouse.StockItems AS si
        ON il.StockItemID = si.StockItemID
        INNER JOIN {_catalog}.Warehouse.PackageTypes AS pt
        ON il.PackageTypeID = pt.PackageTypeID
        INNER JOIN {_catalog}.Sales.Customers AS c
        ON i.CustomerID = c.CustomerID
        INNER JOIN {_catalog}.Sales.Customers AS bt
        ON i.BillToCustomerID = bt.CustomerID
        WHERE CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END > '{last_cutoff}'
        AND CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END <= '{new_cutoff}'
        ORDER BY i.InvoiceID, il.InvoiceLineID
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


    # [delete_existing_target] (DELETE)
    delete_target_sql = f"""
    DELETE FROM {_catalog}.Fact.Sale
    WHERE `WWI Invoice ID` IN (SELECT `WWI Invoice ID` FROM {_catalog}.Integration.Sale_Staging)
    """
    spark.sql(delete_target_sql)


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Fact.Sale AS tgt
        USING (SELECT `Invoice Date Key`, `Delivery Date Key`, `WWI Invoice ID`, `Description`, `Package`, `Quantity`, `Unit Price`, `Tax Rate`, `Total Excluding Tax`, `Tax Amount`, `Profit`, `Total Including Tax`, `Total Dry Items`, `Total Chiller Items`, `WWI City ID`, `WWI Customer ID`, `WWI Bill To Customer ID`, `WWI Stock Item ID`, `WWI Salesperson ID`, `Last Modified When` FROM {_catalog}.Integration.Sale_Staging) AS src
        ON (tgt.`WWI Invoice ID` = src.`WWI Invoice ID` AND tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`)
        WHEN MATCHED AND (src.`Invoice Date Key` IS DISTINCT FROM tgt.`Invoice Date Key` OR src.`Delivery Date Key` IS DISTINCT FROM tgt.`Delivery Date Key` OR src.`Description` IS DISTINCT FROM tgt.`Description` OR src.`Package` IS DISTINCT FROM tgt.`Package` OR src.`Quantity` IS DISTINCT FROM tgt.`Quantity` OR src.`Unit Price` IS DISTINCT FROM tgt.`Unit Price` OR src.`Tax Rate` IS DISTINCT FROM tgt.`Tax Rate` OR src.`Total Excluding Tax` IS DISTINCT FROM tgt.`Total Excluding Tax` OR src.`Tax Amount` IS DISTINCT FROM tgt.`Tax Amount` OR src.`Profit` IS DISTINCT FROM tgt.`Profit` OR src.`Total Including Tax` IS DISTINCT FROM tgt.`Total Including Tax` OR src.`Total Dry Items` IS DISTINCT FROM tgt.`Total Dry Items` OR src.`Total Chiller Items` IS DISTINCT FROM tgt.`Total Chiller Items` OR src.`WWI City ID` IS DISTINCT FROM tgt.`WWI City ID` OR src.`WWI Customer ID` IS DISTINCT FROM tgt.`WWI Customer ID` OR src.`WWI Bill To Customer ID` IS DISTINCT FROM tgt.`WWI Bill To Customer ID` OR src.`WWI Stock Item ID` IS DISTINCT FROM tgt.`WWI Stock Item ID` OR src.`WWI Salesperson ID` IS DISTINCT FROM tgt.`WWI Salesperson ID`)
        THEN UPDATE SET tgt.`Invoice Date Key` = src.`Invoice Date Key`, tgt.`Delivery Date Key` = src.`Delivery Date Key`, tgt.`Description` = src.`Description`, tgt.`Package` = src.`Package`, tgt.`Quantity` = src.`Quantity`, tgt.`Unit Price` = src.`Unit Price`, tgt.`Tax Rate` = src.`Tax Rate`, tgt.`Total Excluding Tax` = src.`Total Excluding Tax`, tgt.`Tax Amount` = src.`Tax Amount`, tgt.`Profit` = src.`Profit`, tgt.`Total Including Tax` = src.`Total Including Tax`, tgt.`Total Dry Items` = src.`Total Dry Items`, tgt.`Total Chiller Items` = src.`Total Chiller Items`, tgt.`WWI City ID` = src.`WWI City ID`, tgt.`WWI Customer ID` = src.`WWI Customer ID`, tgt.`WWI Bill To Customer ID` = src.`WWI Bill To Customer ID`, tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`, tgt.`WWI Salesperson ID` = src.`WWI Salesperson ID`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`Invoice Date Key`, `Delivery Date Key`, `WWI Invoice ID`, `Description`, `Package`, `Quantity`, `Unit Price`, `Tax Rate`, `Total Excluding Tax`, `Tax Amount`, `Profit`, `Total Including Tax`, `Total Dry Items`, `Total Chiller Items`, `WWI City ID`, `WWI Customer ID`, `WWI Bill To Customer ID`, `WWI Stock Item ID`, `WWI Salesperson ID`, `Lineage Key`)
        VALUES (src.`Invoice Date Key`, src.`Delivery Date Key`, src.`WWI Invoice ID`, src.`Description`, src.`Package`, src.`Quantity`, src.`Unit Price`, src.`Tax Rate`, src.`Total Excluding Tax`, src.`Tax Amount`, src.`Profit`, src.`Total Including Tax`, src.`Total Dry Items`, src.`Total Chiller Items`, src.`WWI City ID`, src.`WWI Customer ID`, src.`WWI Bill To Customer ID`, src.`WWI Stock Item ID`, src.`WWI Salesperson ID`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Stock Holding Fact

| Field | Value |
| --- | --- |
| Step ID | `load_stock_holding_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetStockHoldingUpdates` |
| Target Objects | `Integration.StockHolding_Staging` |


In [ ]:
if _step_id in ("all", "load_stock_holding_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.StockHolding_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.StockHolding_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.StockHolding_Staging
    SELECT sih.QuantityOnHand AS `Quantity On Hand`,
               sih.BinLocation AS `Bin Location`,
               sih.LastStocktakeQuantity AS `Last Stocktake Quantity`,
               sih.LastCostPrice AS `Last Cost Price`,
               sih.ReorderLevel AS `Reorder Level`,
               sih.TargetStockLevel AS `Target Stock Level`,
               sih.StockItemID AS `WWI Stock Item ID`
        FROM {_catalog}.Warehouse.StockItemHoldings AS sih
        ORDER BY sih.StockItemID
    """
    spark.sql(staging_extract_sql)


    # [truncate_target] (TRUNCATE)
    spark.sql(f"""
    TRUNCATE TABLE {_catalog}.Fact.Stock Holding
    """)


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Fact.`Stock Holding` AS tgt
        USING (SELECT `Quantity On Hand`, `Bin Location`, `Last Stocktake Quantity`, `Last Cost Price`, `Reorder Level`, `Target Stock Level`, `WWI Stock Item ID` FROM {_catalog}.Integration.StockHolding_Staging) AS src
        ON (tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID`)
        WHEN MATCHED AND (src.`Quantity On Hand` IS DISTINCT FROM tgt.`Quantity On Hand` OR src.`Bin Location` IS DISTINCT FROM tgt.`Bin Location` OR src.`Last Stocktake Quantity` IS DISTINCT FROM tgt.`Last Stocktake Quantity` OR src.`Last Cost Price` IS DISTINCT FROM tgt.`Last Cost Price` OR src.`Reorder Level` IS DISTINCT FROM tgt.`Reorder Level` OR src.`Target Stock Level` IS DISTINCT FROM tgt.`Target Stock Level`)
        THEN UPDATE SET tgt.`Quantity On Hand` = src.`Quantity On Hand`, tgt.`Bin Location` = src.`Bin Location`, tgt.`Last Stocktake Quantity` = src.`Last Stocktake Quantity`, tgt.`Last Cost Price` = src.`Last Cost Price`, tgt.`Reorder Level` = src.`Reorder Level`, tgt.`Target Stock Level` = src.`Target Stock Level`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`WWI Stock Item ID`, `Quantity On Hand`, `Bin Location`, `Last Stocktake Quantity`, `Last Cost Price`, `Reorder Level`, `Target Stock Level`, `Lineage Key`)
        VALUES (src.`WWI Stock Item ID`, src.`Quantity On Hand`, src.`Bin Location`, src.`Last Stocktake Quantity`, src.`Last Cost Price`, src.`Reorder Level`, src.`Target Stock Level`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Stock Item Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_stock_item_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetStockItemUpdates` |
| Target Objects | `Integration.StockItem_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Stock Item Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetStockItemUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedStockItemData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_stock_item_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.StockItem_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.StockItem_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.StockItem_Staging (`WWI Stock Item ID`, `Stock Item`, Color, `Selling Package`, `Buying Package`, Brand, Size, `Lead Time Days`, `Quantity Per Outer`, `Is Chiller Stock`, Barcode, `Tax Rate`, `Unit Price`, `Recommended Retail Price`, `Typical Weight Per Unit`, Photo, `Valid From`, `Valid To`)
        SELECT si.StockItemID, si.StockItemName, c.ColorName, sp.PackageTypeName, bp.PackageTypeName, si.Brand, si.Size, si.LeadTimeDays, si.QuantityPerOuter, si.IsChillerStock, si.Barcode, si.TaxRate, si.UnitPrice, si.RecommendedRetailPrice, si.TypicalWeightPerUnit, si.Photo, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Warehouse.StockItems si
        LEFT OUTER JOIN {_catalog}.Warehouse.Colors c ON si.ColorID = c.ColorID
        LEFT OUTER JOIN {_catalog}.Warehouse.PackageTypes sp ON si.UnitPackageID = sp.PackageTypeID
        LEFT OUTER JOIN {_catalog}.Warehouse.PackageTypes bp ON si.OuterPackageID = bp.PackageTypeID
        WHERE si.LastEditedWhen > '{last_cutoff}' AND si.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Stock Item"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Stock Item"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.`Stock Item` AS tgt
        USING (SELECT `WWI Stock Item ID`, `Stock Item`, Color, `Selling Package`, `Buying Package`, Brand, Size, `Lead Time Days`, `Quantity Per Outer`, `Is Chiller Stock`, Barcode, `Tax Rate`, `Unit Price`, `Recommended Retail Price`, `Typical Weight Per Unit`, Photo, `Valid From`, `Valid To` FROM {_catalog}.Integration.StockItem_Staging) AS src
        ON (tgt.`WWI Stock Item ID` = src.`WWI Stock Item ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.`Stock Item` IS DISTINCT FROM tgt.`Stock Item` OR src.Color IS DISTINCT FROM tgt.Color OR src.`Selling Package` IS DISTINCT FROM tgt.`Selling Package` OR src.`Buying Package` IS DISTINCT FROM tgt.`Buying Package` OR src.Brand IS DISTINCT FROM tgt.Brand OR src.Size IS DISTINCT FROM tgt.Size OR src.`Lead Time Days` IS DISTINCT FROM tgt.`Lead Time Days` OR src.`Quantity Per Outer` IS DISTINCT FROM tgt.`Quantity Per Outer` OR src.`Is Chiller Stock` IS DISTINCT FROM tgt.`Is Chiller Stock` OR src.Barcode IS DISTINCT FROM tgt.Barcode OR src.`Tax Rate` IS DISTINCT FROM tgt.`Tax Rate` OR src.`Unit Price` IS DISTINCT FROM tgt.`Unit Price` OR src.`Recommended Retail Price` IS DISTINCT FROM tgt.`Recommended Retail Price` OR src.`Typical Weight Per Unit` IS DISTINCT FROM tgt.`Typical Weight Per Unit` OR src.Photo IS DISTINCT FROM tgt.Photo)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Stock Item ID`, `Stock Item`, Color, `Selling Package`, `Buying Package`, Brand, Size, `Lead Time Days`, `Quantity Per Outer`, `Is Chiller Stock`, Barcode, `Tax Rate`, `Unit Price`, `Recommended Retail Price`, `Typical Weight Per Unit`, Photo, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Stock Item ID`, src.`Stock Item`, src.Color, src.`Selling Package`, src.`Buying Package`, src.Brand, src.Size, src.`Lead Time Days`, src.`Quantity Per Outer`, src.`Is Chiller Stock`, src.Barcode, src.`Tax Rate`, src.`Unit Price`, src.`Recommended Retail Price`, src.`Typical Weight Per Unit`, src.Photo, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Supplier Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_supplier_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetSupplierUpdates` |
| Target Objects | `Integration.Supplier_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Supplier Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetSupplierUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedSupplierData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_supplier_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Supplier_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Supplier_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Supplier_Staging (`WWI Supplier ID`, Supplier, Category, `Primary Contact`, `Supplier Reference`, `Payment Days`, `Postal Code`, `Valid From`, `Valid To`)
        SELECT s.SupplierID, s.SupplierName, sc.SupplierCategoryName, p.FullName, s.SupplierReference, s.PaymentDays, s.DeliveryPostalCode, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Purchasing.Suppliers s
        LEFT OUTER JOIN {_catalog}.Purchasing.SupplierCategories sc ON s.SupplierCategoryID = sc.SupplierCategoryID
        LEFT OUTER JOIN {_catalog}.Application.People p ON s.PrimaryContactPersonID = p.PersonID
        WHERE s.LastEditedWhen > '{last_cutoff}' AND s.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Supplier"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Supplier"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.Supplier AS tgt
        USING (SELECT `WWI Supplier ID`, Supplier, Category, `Primary Contact`, `Supplier Reference`, `Payment Days`, `Postal Code`, `Valid From`, `Valid To` FROM {_catalog}.Integration.Supplier_Staging) AS src
        ON (tgt.`WWI Supplier ID` = src.`WWI Supplier ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.Supplier IS DISTINCT FROM tgt.Supplier OR src.Category IS DISTINCT FROM tgt.Category OR src.`Primary Contact` IS DISTINCT FROM tgt.`Primary Contact` OR src.`Supplier Reference` IS DISTINCT FROM tgt.`Supplier Reference` OR src.`Payment Days` IS DISTINCT FROM tgt.`Payment Days` OR src.`Postal Code` IS DISTINCT FROM tgt.`Postal Code`)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Supplier ID`, Supplier, Category, `Primary Contact`, `Supplier Reference`, `Payment Days`, `Postal Code`, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Supplier ID`, src.Supplier, src.Category, src.`Primary Contact`, src.`Supplier Reference`, src.`Payment Days`, src.`Postal Code`, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Transaction Fact

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionUpdates` |
| Target Objects | `Integration.Transaction_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Transaction Fact
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['INCREMENTAL_COMPOSITE']
#
# Complexity signals:
# ['UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetTransactionUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedTransactionData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_transaction_fact"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.Transaction_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.Transaction_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.Transaction_Staging
    SELECT CAST(ct.TransactionDate AS date) AS `Date Key`,
               ct.CustomerTransactionID AS `WWI Customer Transaction ID`,
               CAST(NULL AS int) AS `WWI Supplier Transaction ID`,
               ct.InvoiceID AS `WWI Invoice ID`,
               CAST(NULL AS int) AS `WWI Purchase Order ID`,
               CAST(NULL AS STRING) AS `Supplier Invoice Number`,
               ct.AmountExcludingTax AS `Total Excluding Tax`,
               ct.TaxAmount AS `Tax Amount`,
               ct.TransactionAmount AS `Total Including Tax`,
               ct.OutstandingBalance AS `Outstanding Balance`,
               ct.IsFinalized AS `Is Finalized`,
               COALESCE(i.CustomerID, ct.CustomerID) AS `WWI Customer ID`,
               ct.CustomerID AS `WWI Bill To Customer ID`,
               CAST(NULL AS int) AS `WWI Supplier ID`,
               ct.TransactionTypeID AS `WWI Transaction Type ID`,
               ct.PaymentMethodID AS `WWI Payment Method ID`,
               ct.LastEditedWhen AS `Last Modified When`
        FROM {_catalog}.Sales.CustomerTransactions AS ct
        LEFT OUTER JOIN {_catalog}.Sales.Invoices AS i
        ON ct.InvoiceID = i.InvoiceID
        WHERE ct.LastEditedWhen > '{last_cutoff}'
        AND ct.LastEditedWhen <= '{new_cutoff}'

        UNION ALL

        SELECT CAST(st.TransactionDate AS date) AS `Date Key`,
               CAST(NULL AS int) AS `WWI Customer Transaction ID`,
               st.SupplierTransactionID AS `WWI Supplier Transaction ID`,
               CAST(NULL AS int) AS `WWI Invoice ID`,
               st.PurchaseOrderID AS `WWI Purchase Order ID`,
               st.SupplierInvoiceNumber AS `Supplier Invoice Number`,
               st.AmountExcludingTax AS `Total Excluding Tax`,
               st.TaxAmount AS `Tax Amount`,
               st.TransactionAmount AS `Total Including Tax`,
               st.OutstandingBalance AS `Outstanding Balance`,
               st.IsFinalized AS `Is Finalized`,
               CAST(NULL AS int) AS `WWI Customer ID`,
               CAST(NULL AS int) AS `WWI Bill To Customer ID`,
               st.SupplierID AS `WWI Supplier ID`,
               st.TransactionTypeID AS `WWI Transaction Type ID`,
               st.PaymentMethodID AS `WWI Payment Method ID`,
               st.LastEditedWhen AS `Last Modified When`
        FROM {_catalog}.Purchasing.SupplierTransactions AS st
        WHERE st.LastEditedWhen > '{last_cutoff}'
        AND st.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Fact.Transaction AS tgt
        USING (SELECT `Date Key`, `WWI Customer Transaction ID`, `WWI Supplier Transaction ID`, `WWI Invoice ID`, `WWI Purchase Order ID`, `Supplier Invoice Number`, `Total Excluding Tax`, `Tax Amount`, `Total Including Tax`, `Outstanding Balance`, `Is Finalized`, `WWI Customer ID`, `WWI Bill To Customer ID`, `WWI Supplier ID`, `WWI Transaction Type ID`, `WWI Payment Method ID`, `Last Modified When` FROM {_catalog}.Integration.Transaction_Staging) AS src
        ON (tgt.`WWI Customer Transaction ID` = src.`WWI Customer Transaction ID` OR tgt.`WWI Supplier Transaction ID` = src.`WWI Supplier Transaction ID`)
        WHEN MATCHED AND (src.`Date Key` IS DISTINCT FROM tgt.`Date Key` OR src.`Total Excluding Tax` IS DISTINCT FROM tgt.`Total Excluding Tax` OR src.`Tax Amount` IS DISTINCT FROM tgt.`Tax Amount` OR src.`Total Including Tax` IS DISTINCT FROM tgt.`Total Including Tax` OR src.`Outstanding Balance` IS DISTINCT FROM tgt.`Outstanding Balance` OR src.`Is Finalized` IS DISTINCT FROM tgt.`Is Finalized` OR src.`WWI Customer ID` IS DISTINCT FROM tgt.`WWI Customer ID` OR src.`WWI Supplier ID` IS DISTINCT FROM tgt.`WWI Supplier ID` OR src.`WWI Transaction Type ID` IS DISTINCT FROM tgt.`WWI Transaction Type ID` OR src.`WWI Payment Method ID` IS DISTINCT FROM tgt.`WWI Payment Method ID`)
        THEN UPDATE SET tgt.`Date Key` = src.`Date Key`, tgt.`Total Excluding Tax` = src.`Total Excluding Tax`, tgt.`Tax Amount` = src.`Tax Amount`, tgt.`Total Including Tax` = src.`Total Including Tax`, tgt.`Outstanding Balance` = src.`Outstanding Balance`, tgt.`Is Finalized` = src.`Is Finalized`, tgt.`WWI Customer ID` = src.`WWI Customer ID`, tgt.`WWI Supplier ID` = src.`WWI Supplier ID`, tgt.`WWI Transaction Type ID` = src.`WWI Transaction Type ID`, tgt.`WWI Payment Method ID` = src.`WWI Payment Method ID`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED
        THEN INSERT (`Date Key`, `WWI Customer Transaction ID`, `WWI Supplier Transaction ID`, `WWI Invoice ID`, `WWI Purchase Order ID`, `Supplier Invoice Number`, `Total Excluding Tax`, `Tax Amount`, `Total Including Tax`, `Outstanding Balance`, `Is Finalized`, `WWI Customer ID`, `WWI Bill To Customer ID`, `WWI Supplier ID`, `WWI Transaction Type ID`, `WWI Payment Method ID`, `Lineage Key`)
        VALUES (src.`Date Key`, src.`WWI Customer Transaction ID`, src.`WWI Supplier Transaction ID`, src.`WWI Invoice ID`, src.`WWI Purchase Order ID`, src.`Supplier Invoice Number`, src.`Total Excluding Tax`, src.`Tax Amount`, src.`Total Including Tax`, src.`Outstanding Balance`, src.`Is Finalized`, src.`WWI Customer ID`, src.`WWI Bill To Customer ID`, src.`WWI Supplier ID`, src.`WWI Transaction Type ID`, src.`WWI Payment Method ID`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Transaction Type Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_type_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionTypeUpdates` |
| Target Objects | `Integration.TransactionType_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Transaction Type Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetTransactionTypeUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedTransactionTypeData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
if _step_id in ("all", "load_transaction_type_dimension"):
    # [set_table_name] (CONTROL)
    table_name = f"{_catalog}.Integration.TransactionType_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql(f"""
    DELETE FROM {_catalog}.Integration.TransactionType_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    staging_extract_sql = f"""
    INSERT INTO {_catalog}.Integration.TransactionType_Staging (`WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`)
        SELECT tt.TransactionTypeID, tt.TransactionTypeName, '{last_cutoff}', '{new_cutoff}'
        FROM {_catalog}.Application.TransactionTypes tt
        WHERE tt.LastEditedWhen > '{last_cutoff}' AND tt.LastEditedWhen <= '{new_cutoff}'
    """
    spark.sql(staging_extract_sql)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Transaction Type"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Transaction Type"


    # [insert_target] (INSERT)
    insert_target_sql = f"""
    MERGE INTO {_catalog}.Dimension.`Transaction Type` AS tgt
        USING (SELECT `WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To` FROM {_catalog}.Integration.TransactionType_Staging) AS src
        ON (tgt.`WWI Transaction Type ID` = src.`WWI Transaction Type ID` AND tgt.`Valid To` = src.`Valid From`)
        WHEN MATCHED AND (src.`Transaction Type` IS DISTINCT FROM tgt.`Transaction Type`)
        THEN UPDATE SET tgt.`Valid To` = src.`Valid To`, tgt.`Lineage Key` = {lineage_key}
        WHEN NOT MATCHED AND src.`Valid To` = '9999-12-31 23:59:59.9999999'
        THEN INSERT (`WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`, `Lineage Key`)
        VALUES (src.`WWI Transaction Type ID`, src.`Transaction Type`, src.`Valid From`, src.`Valid To`, {lineage_key})
    """
    spark.sql(insert_target_sql)


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )
